# Walkie Causal LM 完整模型

源码导航：[`core/model/walkie.py`](../../../core/model/walkie.py) 中的 `WalkieConfig`、`WalkieBlock`、`WalkieForCausalLM`。

Walkie 的模型主体是一个 decoder-only Causal LM，在 GPT-2 教学实现的基础上替换了全部核心子模块，以支持高效的 1B 量级训练与推断。

### 1. 架构综述

整体数据流：

$$
\mathbf{h}_0 = \text{Embedding}(\text{idx})
$$
$$
\mathbf{h}_{l+1} = \text{WalkieBlock}_l(\mathbf{h}_l), \quad l = 0, \ldots, L-1
$$
$$
\text{logits} = \text{LM-Head}(\text{RMSNorm}(\mathbf{h}_L))
$$

**WalkieBlock 的 Pre-norm 残差结构**（与 Post-norm 不同，归一化在残差支路之前）：

$$
\mathbf{x}' = \mathbf{x} + \text{Attn}(\text{RMSNorm}(\mathbf{x}))
$$
$$
\mathbf{x}'' = \mathbf{x}' + \text{SwiGLU}(\text{RMSNorm}(\mathbf{x}'))
$$

与 GPT-2 基础实现的主要差异：

| 组件 | GPT-2 baseline | Walkie |
|---|---|---|
| 归一化 | LayerNorm（post-norm） | RMSNorm（pre-norm） |
| 注意力 | MHA，Learned Pos Emb | GQA + QK-Norm + RoPE |
| FFN | GELU MLP | SwiGLU（无 bias） |
| Weight tying | 否 | 是（默认，节省约 100M 参数） |
| Loss 计算 | 全量 logits | Chunked Cross-Entropy |

### 2. Chunked Cross-Entropy 与显存优化

LM-head 将 $\mathbf{h} \in \mathbb{R}^{B \times T \times d}$ 投影到 $\mathbb{R}^{B \times T \times V}$（$V = 65536$），这一中间张量在 $B=4, T=16384$ 时约占 **16 GB**（fp32）。Walkie 将序列维度分块计算 loss：

```python
for start in range(0, T, chunk_size):
    logits = lm_head(hidden[:, start:stop, :])  # 仅存 chunk_size 的 logits
    loss += cross_entropy(logits, targets[:, start:stop], reduction='sum')
loss /= total_valid_tokens
```

仅保留当前块的 logits，其余被释放，显存峰值降低至 $O(\text{chunk\_size} \times V)$。

### 3. Weight Tying

`lm_head.weight = tok_embeddings.weight`：LM head 的参数矩阵直接复用 token embedding 矩阵。对词表大小 $V = 65536$、隐维度 $d = 1536$ 的模型，这节省约 $65536 \times 1536 \times 4\text{B} \approx 402\text{MB}$ 的参数存储，使总参数量控制在 1B 以内。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.model.walkie import WalkieConfig, WalkieForCausalLM

### 4. Tiny 配置前向 / loss / generate 验证

In [ ]:
cfg = WalkieConfig(
    vocab_size=128,
    block_size=64,
    n_embd=64,
    n_layer=2,
    n_head=4,
    n_head_kv=2,
    head_dim=16,
    d_ffn=128,
    dropout=0.0,
    bias=False,
    tie_weights=True,
)
model = WalkieForCausalLM(cfg)

idx     = torch.randint(0, cfg.vocab_size, (2, 8))
targets = torch.randint(0, cfg.vocab_size, (2, 8))
logits, loss = model(idx, targets)

print("logits shape :", tuple(logits.shape))  # (2, 8, vocab_size)
print("loss         :", round(float(loss), 4))
print("params       :", model.num_parameters(), "(weight tying 只计 1 次 embedding)")
print("weight tied  :", model.lm_head.weight.data_ptr() == model.tok_embeddings.weight.data_ptr())

# 贪心 / 温度采样生成
generated = model.generate(idx[:1], max_new_tokens=4, temperature=0.0)
print("generated    :", tuple(generated.shape))  # (1, 12)

### 5. 1B 生产配置的参数量估算

In [ ]:
cfg = WalkieConfig()
V, D, L = cfg.vocab_size, cfg.n_embd, cfg.n_layer
H, Hkv, Hd, F = cfg.n_head, cfg.n_head_kv, cfg.head_dim, cfg.d_ffn

# Embedding（weight tying 下 lm_head 不计入）
embed = V * D

# 每 WalkieBlock：
#   Attn  = q_proj + k_proj + v_proj + o_proj
#   SwiGLU= gate_proj + up_proj + down_proj
#   Norms = 2 × RMSNorm (n_embd) + 2 × QK-Norm (head_dim)
attn_params  = D * H * Hd + 2 * D * Hkv * Hd + H * Hd * D
ffn_params   = 2 * D * F + F * D
norm_params  = 2 * D + 2 * Hd  # attn_norm + ffn_norm + q_norm + k_norm (per head dim)
per_layer    = attn_params + ffn_params + norm_params

# 最终 norm
final_norm = D

total = embed + L * per_layer + final_norm

print(f"Embedding      : {embed / 1e6:.1f} M")
print(f"Per Layer      : {per_layer / 1e6:.3f} M  × {L} layers")
print(f"Estimated total: {total / 1e6:.1f} M")
print(f"Under 1B       : {total < 1_000_000_000}")

# 实例化真实模型对比
real_model = WalkieForCausalLM(WalkieConfig())
print(f"Actual params  : {real_model.num_parameters() / 1e6:.1f} M")

### 6. 源码精讲

**WalkieBlock**（`core/model/walkie.py`）：

```python
class WalkieBlock(nn.Module):
    """Pre-norm 残差块: x = x + Attn(RMSNorm(x)); x = x + SwiGLU(RMSNorm(x))."""

    def __init__(self, cfg, rope):
        super().__init__()
        self.norm_attn = RMSNorm(cfg.n_embd, eps=cfg.rms_norm_eps)  # 注意力前归一化
        self.attn = WalkieCausalSelfAttention(...)                   # GQA+QK-Norm+RoPE
        self.norm_ffn = RMSNorm(cfg.n_embd, eps=cfg.rms_norm_eps)   # FFN 前归一化
        self.mlp = SwiGLUMLP(...)                                    # SwiGLU FFN

    def forward(self, x):
        x = x + self.attn(self.norm_attn(x))   # 残差 1：注意力支路
        x = x + self.mlp(self.norm_ffn(x))     # 残差 2：FFN 支路
        return x
```

**残差输出投影的 scaled init**（`WalkieForCausalLM.__init__`）：

```python
for pn, p in self.named_parameters():
    if pn.endswith("o_proj.weight") or pn.endswith("down_proj.weight"):
        # 残差路径上的输出投影乘以 1/sqrt(2 * n_layer) 防止深层梯度爆炸
        nn.init.normal_(p, mean=0.0, std=cfg.init_std / math.sqrt(2 * cfg.n_layer))
```

这与 GPT-2 原始技术报告中的做法一致：深度越大，残差输出越需要缩放以维持训练稳定性。

---

## 延伸阅读与参考资料

### 核心论文
- **GPT-2**: Radford et al., 2019. [report](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)
- **DeepSeek-Coder**: Guo et al., 2024. [arXiv:2401.14196](https://arxiv.org/abs/2401.14196)

### 工程实现
- **nanoGPT**: [GitHub](https://github.com/karpathy/nanoGPT)
- **Hugging Face causal LM implementations**: [source](https://github.com/huggingface/transformers/tree/main/src/transformers/models)